# ANN FOR CLASSIFICATION PROBLEM

In [66]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [67]:
# Loading the data
data = pd.read_csv("DateFruit_Dataset.csv")
data.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [68]:
x = data.drop("Class", axis =1)
y = data["Class"]
# labling the output class
le = LabelEncoder()
y = le.fit_transform(y)


In [69]:
# spliting and Scaling the data
x_train, x_test, y_train, y_test = train_test_split(x,y, random_state=42, test_size=0.2)
sc = StandardScaler()
x_train_scaled = sc.fit_transform(x_train)
x_test_scaled = sc.fit_transform(x_test)

In [70]:
# Building the dataloeader and Tensordata
x_train_tensor = torch.tensor(x_train_scaled, dtype= torch.float32)
x_test_tensor = torch.tensor(x_test_scaled, dtype= torch.float32)
y_train_tensor = torch.tensor(y_train, dtype= torch.long)
y_test_tensor = torch.tensor(y_test, dtype= torch.long)

train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
test_dataset = TensorDataset(x_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)


# BUILDING THE CLASSIFICATION MODEL

In [71]:
# Build our model
class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(x_train.shape[1], 64),
            nn.ReLU(),
            nn.Linear(64,64),
            nn.ReLU(),
            nn.Linear(64, 7),
            )
    def forward(self,x):
        return self.model(x)


In [72]:
model = ANN()
criterian = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [73]:
# Training the NN
epocs = 100
for epoc in range(epocs):
    model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterian(outputs, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss
        
    running_loss= running_loss/len(train_loader)
    print(f"epocs = {epoc +1} / {epocs}: loss is {running_loss}")

epocs = 1 / 100: loss is 1.7126258611679077
epocs = 2 / 100: loss is 1.15132474899292
epocs = 3 / 100: loss is 0.732031524181366
epocs = 4 / 100: loss is 0.5447182655334473
epocs = 5 / 100: loss is 0.44581466913223267
epocs = 6 / 100: loss is 0.38878777623176575
epocs = 7 / 100: loss is 0.34584853053092957
epocs = 8 / 100: loss is 0.3158707916736603
epocs = 9 / 100: loss is 0.3011842370033264
epocs = 10 / 100: loss is 0.2750326097011566
epocs = 11 / 100: loss is 0.2512054443359375
epocs = 12 / 100: loss is 0.22712907195091248
epocs = 13 / 100: loss is 0.21830379962921143
epocs = 14 / 100: loss is 0.2077004611492157
epocs = 15 / 100: loss is 0.19278505444526672
epocs = 16 / 100: loss is 0.18113353848457336
epocs = 17 / 100: loss is 0.18841171264648438
epocs = 18 / 100: loss is 0.18083931505680084
epocs = 19 / 100: loss is 0.16665729880332947
epocs = 20 / 100: loss is 0.1653517633676529
epocs = 21 / 100: loss is 0.15690995752811432
epocs = 22 / 100: loss is 0.15107758343219757
epocs = 23

In [75]:
# Evaluation
model.eval()
total = 0
correct = 0
with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb) # [0.2,0.5,1.3.... ] 7 values
        _,predicted = torch.max(outputs,1) # this gives us the index and the maximum output value with each output
        correct += (predicted == yb.squeeze()).sum().item()
        total += yb.size(0)  # actual Sample in each batch

    print(f"Accuracy >> {correct/total *100} ")

Accuracy >> 93.33333333333333 
